# Day 3 - Exploratory Data Analysis

This notebook explores the cleaned mutual fund datasets created in Day 2 and exports all charts to `reports/charts/`.


In [2]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120

project_root = None
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / 'data' / 'processed').exists():
        project_root = candidate
        break
if project_root is None:
    project_root = Path.cwd().resolve()

data_dir = project_root / 'data' / 'processed'
charts_dir = project_root / 'reports' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)

def load_csv(name, parse_dates=None):
    return pd.read_csv(data_dir / name, parse_dates=parse_dates)

fund_master = load_csv('01_fund_master_cleaned.csv', parse_dates=['launch_date'])
nav = load_csv('02_nav_history_cleaned.csv', parse_dates=['date'])
aum = load_csv('03_aum_by_fund_house_cleaned.csv', parse_dates=['date'])
sip = load_csv('04_monthly_sip_inflows_cleaned.csv')
sip['month'] = pd.to_datetime(sip['month'] + '-01')
category_inflows = load_csv('05_category_inflows_cleaned.csv')
category_inflows['month'] = pd.to_datetime(category_inflows['month'] + '-01')
folio = load_csv('06_industry_folio_count_cleaned.csv')
folio['month'] = pd.to_datetime(folio['month'] + '-01')
performance = load_csv('07_scheme_performance_cleaned.csv')
transactions = load_csv('08_investor_transactions_cleaned.csv', parse_dates=['transaction_date'])
holdings = load_csv('09_portfolio_holdings_cleaned.csv', parse_dates=['portfolio_date'])
benchmark = load_csv('10_benchmark_indices_cleaned.csv', parse_dates=['date'])

scheme_names = fund_master[['amfi_code', 'scheme_name', 'fund_house']].drop_duplicates('amfi_code')
nav = nav.merge(scheme_names, on='amfi_code', how='left')
performance = performance.merge(scheme_names, on='amfi_code', how='left', suffixes=('', '_master'))

transactions['transaction_type'] = transactions['transaction_type'].astype(str)
transactions['age_group'] = transactions['age_group'].astype(str)
transactions['city_tier'] = transactions['city_tier'].astype(str)
transactions['state'] = transactions['state'].astype(str)
transactions['gender'] = transactions['gender'].astype(str)

sip_transactions = transactions[transactions['transaction_type'] == 'SIP'].copy()

def save_plotly(fig, filename):
    path = charts_dir / filename
    fig.write_image(str(path), scale=2)
    print(f'Saved {path.name}')

def save_matplotlib(fig, filename):
    path = charts_dir / filename
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path.name}')

def ordered_age_groups(series):
    preferred = ['18-25', '26-35', '36-45', '46-55', '56+']
    present = [item for item in preferred if item in set(series.astype(str))]
    leftovers = sorted([item for item in series.astype(str).unique() if item not in present])
    return present + leftovers


In [3]:
# Chart 1: NAV trend analysis for all 40 schemes with 2023 bull run and 2024 correction highlights
nav_trend = nav.sort_values(['amfi_code', 'date']).copy()
nav_trend['nav_index'] = nav_trend.groupby('amfi_code')['nav'].transform(lambda s: s / s.iloc[0] * 100)
avg_nav = nav_trend.groupby('date', as_index=False)['nav_index'].mean()

fig = px.line(
    nav_trend,
    x='date',
    y='nav_index',
    color='scheme_name',
    title='NAV Trend Analysis (Normalized Index = 100 at Start) Across All 40 Schemes',
    labels={'date': 'Date', 'nav_index': 'NAV Index (Base 100)', 'scheme_name': 'Scheme'},
    render_mode='svg',
)
fig.update_traces(opacity=0.18, line={'width': 1}, showlegend=False)
fig.add_trace(
    go.Scatter(
        x=avg_nav['date'],
        y=avg_nav['nav_index'],
        mode='lines',
        name='Average NAV Index',
        line={'color': '#ff7f0e', 'width': 4},
    )
)
fig.add_vrect(x0='2023-01-01', x1='2023-12-31', fillcolor='green', opacity=0.12, line_width=0)
fig.add_vrect(x0='2024-01-01', x1='2024-12-31', fillcolor='red', opacity=0.12, line_width=0)
fig.add_annotation(x='2023-06-15', y=nav_trend['nav_index'].max() * 0.98, text='2023 bull run', showarrow=False, font={'color': 'green'})
fig.add_annotation(x='2024-06-15', y=nav_trend['nav_index'].max() * 0.90, text='2024 correction', showarrow=False, font={'color': 'red'})
fig.update_layout(template='plotly_white', height=700, width=1500)
save_plotly(fig, '01_nav_trend_all_schemes.png')
fig.show()


Saved 01_nav_trend_all_schemes.png


**Insight 1:** The NAV trend chart shows that the scheme universe recovered strongly during the 2023 bull phase, while the 2024 shaded period highlights a broad correction and divergence in scheme trajectories. See `../reports/charts/01_nav_trend_all_schemes.png`.


In [4]:
# Chart 2: AUM growth grouped bar chart (2022-2025)
aum_yearly = aum.copy()
aum_yearly['year'] = aum_yearly['date'].dt.year
aum_yearly = aum_yearly[aum_yearly['year'].between(2022, 2025)]
aum_yearly = aum_yearly.groupby(['year', 'fund_house'], as_index=False)['aum_crore'].mean()
top_funds = aum_yearly.groupby('fund_house')['aum_crore'].mean().nlargest(8).index
aum_plot = aum_yearly[aum_yearly['fund_house'].isin(top_funds)].copy()

fig, ax = plt.subplots(figsize=(16, 7))
sns.barplot(data=aum_plot, x='year', y='aum_crore', hue='fund_house', ax=ax)
ax.set_title('AUM Growth by Fund House (2022-2025)', pad=16)
ax.set_xlabel('Year')
ax.set_ylabel('Average AUM (Crore INR)')
ax.legend(title='Fund House', bbox_to_anchor=(1.02, 1), loc='upper left')
save_matplotlib(fig, '02_aum_growth_grouped_bar.png')
plt.show()


Saved 02_aum_growth_grouped_bar.png


**Insight 2:** AUM growth is concentrated in a handful of large fund houses, and the year-wise grouping makes the expansion trend visible across the 2022–2025 period. See `../reports/charts/02_aum_growth_grouped_bar.png`.


In [5]:
# Chart 3: Monthly SIP inflow time-series with peak annotation
sip_sorted = sip.sort_values('month').copy()
peak_row = sip_sorted.loc[sip_sorted['sip_inflow_crore'].idxmax()]

fig = px.line(
    sip_sorted,
    x='month',
    y='sip_inflow_crore',
    title='Monthly SIP Inflows (Jan 2022 - Dec 2025)',
    labels={'month': 'Month', 'sip_inflow_crore': 'SIP Inflow (Crore INR)'},
    markers=True,
)
fig.add_annotation(
    x=peak_row['month'].strftime("%Y-%m") if hasattr(peak_row['month'], "strftime") else str(peak_row['month']),
    y=peak_row['sip_inflow_crore'],
    text=f"Peak: ₹{peak_row['sip_inflow_crore']:,.0f} Cr",
    showarrow=True,
    arrowhead=2,
    ax=20,
    ay=-40,
)
fig.add_hline(y=peak_row['sip_inflow_crore'], line_dash='dash', line_color='orange')
fig.update_layout(template='plotly_white', height=650, width=1400)
save_plotly(fig, '03_monthly_sip_inflow_trend.png')
fig.show()


Saved 03_monthly_sip_inflow_trend.png


**Insight 3:** SIP inflows show a persistent upward pattern with a clear peak around ₹31,002 Cr, which supports the story of growing retail participation in the market. See `../reports/charts/03_monthly_sip_inflow_trend.png`.


In [6]:
# Chart 4: Category inflow heatmap
category_pivot = category_inflows.pivot_table(
    index='category',
    columns='month',
    values='net_inflow_crore',
    aggfunc='sum',
).fillna(0)
category_pivot = category_pivot.loc[category_pivot.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(category_pivot, cmap='YlGnBu', linewidths=0.3, ax=ax)
ax.set_title('Category Inflow Heatmap', pad=16)
ax.set_xlabel('Month')
ax.set_ylabel('Category')
save_matplotlib(fig, '04_category_inflow_heatmap.png')
plt.show()


Saved 04_category_inflow_heatmap.png


**Insight 4:** The heatmap reveals that liquidity-oriented and large-scale categories attract the strongest and most persistent inflows, while some categories remain cyclical. See `../reports/charts/04_category_inflow_heatmap.png`.


In [7]:
# Chart 5: Age-group pie chart
age_counts = transactions['age_group'].value_counts().reindex(ordered_age_groups(transactions['age_group']), fill_value=0)
age_labels = [str(label) for label in age_counts.index]
age_values = [float(value) for value in age_counts.values]
fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(age_values, labels=age_labels, autopct='%1.1f%%', startangle=90, colors=sns.color_palette('Set2', len(age_counts)))
ax.set_title('Investor Age-Group Distribution', pad=18)
save_matplotlib(fig, '05_age_group_pie.png')
plt.show()


Saved 05_age_group_pie.png


**Insight 5:** The age-group split confirms that participation is broad-based, with the middle age cohorts carrying the largest share of activity. See `../reports/charts/05_age_group_pie.png`.


In [8]:
# Chart 6: SIP amount boxplot by age group
sip_age = sip_transactions.copy()
sip_age = sip_age[sip_age['age_group'].isin(ordered_age_groups(sip_age['age_group']))]

fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(data=sip_age, x='age_group', y='amount_inr', order=ordered_age_groups(sip_age['age_group']), ax=ax)
ax.set_title('SIP Amount Distribution by Age Group', pad=16)
ax.set_xlabel('Age Group')
ax.set_ylabel('SIP Amount (INR)')
ax.tick_params(axis='x', rotation=0)
save_matplotlib(fig, '06_sip_amount_boxplot_by_age_group.png')
plt.show()


Saved 06_sip_amount_boxplot_by_age_group.png


**Insight 6:** SIP ticket sizes vary materially by age group, with some older cohorts showing a wider and higher-value investment distribution. See `../reports/charts/06_sip_amount_boxplot_by_age_group.png`.


In [9]:
# Chart 7: Gender distribution
gender_counts = transactions['gender'].value_counts()
gender_labels = [str(label) for label in gender_counts.index]
gender_values = [float(value) for value in gender_counts.values]
fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(gender_values, labels=gender_labels, autopct='%1.1f%%', startangle=90, colors=sns.color_palette('pastel', len(gender_counts)))
ax.set_title('Investor Gender Distribution', pad=18)
save_matplotlib(fig, '07_gender_distribution.png')
plt.show()


Saved 07_gender_distribution.png


**Insight 7:** Gender distribution remains uneven, but the chart provides a quick segmentation view for understanding who is currently most active in the dataset. See `../reports/charts/07_gender_distribution.png`.


In [10]:
# Chart 8: SIP amount by state
state_sip = sip_transactions.groupby('state', as_index=False)['amount_inr'].sum().sort_values('amount_inr', ascending=False).head(15)
state_sip = state_sip.sort_values('amount_inr', ascending=True)

fig, ax = plt.subplots(figsize=(14, 8))
sns.barplot(data=state_sip, y='state', x='amount_inr', ax=ax, palette='Blues_r')
ax.set_title('Top States by SIP Amount', pad=16)
ax.set_xlabel('SIP Amount (INR)')
ax.set_ylabel('State')
save_matplotlib(fig, '08_statewise_sip_amount.png')
plt.show()


C:\Users\karth\AppData\Local\Temp\ipykernel_12980\3839378477.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=state_sip, y='state', x='amount_inr', ax=ax, palette='Blues_r')


Saved 08_statewise_sip_amount.png


**Insight 8:** SIP flows are geographically concentrated, with a few large states accounting for a disproportionate share of invested value. See `../reports/charts/08_statewise_sip_amount.png`.


In [11]:
# Chart 9: T30 vs B30 pie chart
tier_amounts = sip_transactions.groupby('city_tier', as_index=False)['amount_inr'].sum()
tier_amounts = tier_amounts[tier_amounts['city_tier'].isin(['T30', 'B30'])]
tier_amounts = tier_amounts.set_index('city_tier').reindex(['T30', 'B30']).fillna(0)

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(
    tier_amounts['amount_inr'],
    labels=tier_amounts.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=['#4C78A8', '#F58518'],
)
ax.set_title('T30 vs B30 SIP Amount Split', pad=18)
save_matplotlib(fig, '09_t30_vs_b30_pie.png')
plt.show()


Saved 09_t30_vs_b30_pie.png


**Insight 9:** The T30 vs B30 mix helps frame the spread between metro and non-metro participation, which is essential for investor demographic analysis. See `../reports/charts/09_t30_vs_b30_pie.png`.


In [12]:
# Chart 9: T30 vs B30 pie chart
tier_amounts = sip_transactions.groupby('city_tier', as_index=False)['amount_inr'].sum()
tier_amounts = tier_amounts[tier_amounts['city_tier'].isin(['T30', 'B30'])]
tier_amounts = tier_amounts.set_index('city_tier').reindex(['T30', 'B30']).fillna(0)
tier_labels = [str(label) for label in tier_amounts.index]
tier_values = [float(value) for value in tier_amounts['amount_inr']]

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(
    tier_values,
    labels=tier_labels,
    autopct='%1.1f%%',
    startangle=90,
    colors=['#4C78A8', '#F58518'],
)
ax.set_title('T30 vs B30 SIP Amount Split', pad=18)
save_matplotlib(fig, '09_t30_vs_b30_pie.png')
plt.show()


Saved 09_t30_vs_b30_pie.png


**Insight 10:** Folio growth shows a sustained rise in investor participation, and the milestone annotations make the acceleration easy to spot over time. See `../reports/charts/10_folio_count_growth_line.png`.


In [13]:
# Chart 11: NAV return correlation heatmap for 10 selected funds
top10_codes = performance.nlargest(10, 'aum_crore')['amfi_code'].tolist()
nav_10 = nav[nav['amfi_code'].isin(top10_codes)][['date', 'amfi_code', 'nav', 'scheme_name']].drop_duplicates()
nav_10_pivot = nav_10.pivot_table(index='date', columns='scheme_name', values='nav').sort_index()
returns_10 = nav_10_pivot.pct_change().dropna(how='all')
corr_10 = returns_10.corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_10, cmap='coolwarm', center=0, annot=True, fmt='.2f', ax=ax)
ax.set_title('NAV Return Correlation Heatmap - Top 10 Funds by AUM', pad=16)
save_matplotlib(fig, '11_nav_return_correlation_heatmap.png')
plt.show()


Saved 11_nav_return_correlation_heatmap.png


In [14]:
# Chart 12: Sector allocation donut chart
sector_allocation = holdings.groupby('sector', as_index=False)['market_value_cr'].sum().sort_values('market_value_cr', ascending=False)
fig = go.Figure(
    data=[
        go.Pie(
            labels=sector_allocation['sector'],
            values=sector_allocation['market_value_cr'],
            hole=0.45,
            textinfo='label+percent',
        )
    ]
)
fig.update_layout(title='Sector Allocation Donut Chart', template='plotly_white', width=1000, height=800)
save_plotly(fig, '12_sector_allocation_donut.png')
fig.show()


Saved 12_sector_allocation_donut.png


In [15]:
# Chart 13: Top 10 funds by 3-year return
top_returns = performance[['scheme_name', 'return_3yr_pct', 'category', 'fund_house']].drop_duplicates().sort_values('return_3yr_pct', ascending=False).head(10)
fig, ax = plt.subplots(figsize=(14, 8))
sns.barplot(data=top_returns, y='scheme_name', x='return_3yr_pct', ax=ax, palette='viridis')
ax.set_title('Top 10 Funds by 3-Year Return', pad=16)
ax.set_xlabel('3-Year Return (%)')
ax.set_ylabel('Scheme')
save_matplotlib(fig, '13_top_10_funds_by_3yr_return.png')
plt.show()


C:\Users\karth\AppData\Local\Temp\ipykernel_12980\2708536889.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=top_returns, y='scheme_name', x='return_3yr_pct', ax=ax, palette='viridis')


Saved 13_top_10_funds_by_3yr_return.png


In [16]:
# Chart 12: Sector allocation donut chart
sector_allocation = (
    holdings.groupby('sector', as_index=False)[['market_value_cr']]
    .sum()
    .sort_values(by='market_value_cr', ascending=False)
)
sector_labels = [str(label) for label in sector_allocation['sector']]
sector_values = [float(value) for value in sector_allocation['market_value_cr']]
fig = go.Figure(
    data=[
        go.Pie(
            labels=sector_labels,
            values=sector_values,
            hole=0.45,
            textinfo='label+percent',
        )
    ]
)
fig.update_layout(title='Sector Allocation Donut Chart', template='plotly_white', width=1000, height=800)
save_plotly(fig, '12_sector_allocation_donut.png')
fig.show()


Saved 12_sector_allocation_donut.png


In [17]:
# Chart 15: Transaction type distribution
txn_distribution = transactions['transaction_type'].value_counts()
txn_labels = [str(label) for label in txn_distribution.index]
txn_values = [float(value) for value in txn_distribution.values]
fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(
    txn_values,
    labels=txn_labels,
    autopct='%1.1f%%',
    startangle=90,
    colors=sns.color_palette('Set3', len(txn_distribution)),
)
ax.set_title('Transaction Type Distribution', pad=18)
save_matplotlib(fig, '15_transaction_type_distribution.png')
plt.show()


Saved 15_transaction_type_distribution.png


In [18]:
# Final verification of exported charts
generated_pngs = sorted(charts_dir.glob('*.png'))
print(f'Generated {len(generated_pngs)} PNG charts in {charts_dir}')
for png in generated_pngs:
    print(png.name)


Generated 15 PNG charts in c:\Users\karth\bluestock_mf_capstone\reports\charts
01_nav_trend_all_schemes.png
02_aum_growth_grouped_bar.png
03_monthly_sip_inflow_trend.png
04_category_inflow_heatmap.png
05_age_group_pie.png
06_sip_amount_boxplot_by_age_group.png
07_gender_distribution.png
08_statewise_sip_amount.png
09_t30_vs_b30_pie.png
10_folio_count_growth_line.png
11_nav_return_correlation_heatmap.png
12_sector_allocation_donut.png
13_top_10_funds_by_3yr_return.png
14_expense_ratio_vs_sharpe_scatter.png
15_transaction_type_distribution.png
